# Project Canary — Day 35 Average-Weight Forecast

**Purpose:** reproduce the complete weight-model workflow in a form the capstone team can run and defend.

**Business question:** given the weights recorded so far, what average building weight should we expect on Day 35, compared with the 1,800 g milestone?

## 1. Define Y, X, and the unit of analysis

- **Y target:** observed building average bodyweight on production Day 35.
- **One independent outcome:** one building in one cycle with a Day 35 measurement.
- **Training rows:** up to four checkpoint views of that outcome—Day 7, 14, 21, and 28. These are repeated views, not 124 independent flocks.
- **X inputs for Ridge:** measurement day; latest weight; weight ÷ the interpolated farm target for that day; recent and cumulative average daily gain; and the Day 7/14/21/28 checkpoint weights known by that review date.
- The interpolated 1,800 g target curve is an input/reference. It is **not** the Y label and does not manufacture an actual Day 35 result.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "canary").exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "FARM HARVEST DATA.xlsx"
MODEL_READY_DIR = ROOT / "outputs" / "model_ready"

from canary import load_workbook

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
dataset = load_workbook(DATA_PATH)
print(f"Source: {dataset.source_name}")
print(f"Canonical building-day rows: {len(dataset.daily):,}")
print(f"Recorded building-cycles: {len(dataset.cycles):,}")
print(f"Blocking data-quality checks passed: {dataset.quality.passed}")
print(f"Non-blocking warnings: {len(dataset.quality.warnings)}")

Source: FARM HARVEST DATA.xlsx
Canonical building-day rows: 1,666
Recorded building-cycles: 34
Blocking data-quality checks passed: True
Non-blocking warnings: 3


In [2]:
from canary import build_day35_feature_rows, build_day35_training_rows, train_day35_weight_baseline

rows = build_day35_training_rows(dataset)
engineered_rows = build_day35_feature_rows(dataset)
outcomes = rows[["cycle_id", "building_id", "actual_day35_weight_kg"]].drop_duplicates()
coverage = pd.DataFrame({
    "Measure": ["Historical cycles", "Distinct Day 35 building outcomes", "Checkpoint training rows", "At/above 1,800 g", "Below 1,800 g"],
    "Count": [rows["cycle_id"].nunique(), len(outcomes), len(rows), outcomes["actual_day35_weight_kg"].ge(1.8).sum(), outcomes["actual_day35_weight_kg"].lt(1.8).sum()],
})
coverage

,Measure,Count
0,Historical cycles,6
1,Distinct Day 35 building outcomes,31
2,Checkpoint training rows,124
3,"At/above 1,800 g",5
4,"Below 1,800 g",26


In [3]:
exported = pd.read_csv(MODEL_READY_DIR / "day35_weight_training.csv")
assert len(exported) == len(engineered_rows) == 124
shared = [column for column in engineered_rows.columns if column in exported.columns]
left = engineered_rows[shared].copy().sort_values(["cycle_id", "building_id", "measurement_day"]).reset_index(drop=True)
right = exported[shared].copy().sort_values(["cycle_id", "building_id", "measurement_day"]).reset_index(drop=True)
for column in left.columns:
    if pd.api.types.is_numeric_dtype(left[column]):
        assert np.allclose(left[column], pd.to_numeric(right[column]), equal_nan=True)
    else:
        assert left[column].astype(str).equals(right[column].astype(str))
print("Export reconciliation passed: the CSV contains the exact 124 engineered weight rows.")

Export reconciliation passed: the CSV contains the exact 124 engineered weight rows.


## 2. Preprocessing and validation

1. Correct weight rows during workbook standardization and aggregate zone records to one building-day.
2. Keep only observed checkpoint weights and observed Day 35 labels; never fill a missing Day 35 Y from the target curve.
3. At each checkpoint, hide future checkpoint weights.
4. Median-impute missing X values inside the training fold; Ridge inputs are standardized.
5. Use **leave-one-complete-cycle-out cross-validation** so every test prediction comes from a model that never saw that cycle.
6. Optimize **cycle-macro MAE in kilograms**. Choose the simplest model within 5% of the best to avoid rewarding tiny unstable gains.

In [4]:
manifest = train_day35_weight_baseline(dataset)
print("Champion:", manifest["selected_model"])
print("Model version:", manifest["model_version"])
print("Selected X inputs:")
for feature in manifest["ridge_parameters"]["features"]:
    print(" -", feature)

Champion: ridge_regression
Model version: day35-weight-0.4.0
Selected X inputs:
 - measurement_day
 - current_weight_kg
 - current_to_target_ratio
 - recent_adg_kg_day
 - has_recent_adg
 - cumulative_adg_kg_day
 - weight_day_7_kg
 - weight_day_14_kg
 - weight_day_21_kg
 - weight_day_28_kg


## 3. Candidate comparison

In [5]:
comparison = pd.DataFrame([
    {
        "Candidate": name,
        "MAE (g)": metrics["mae_kg"] * 1000,
        "Cycle-macro MAE (g)": metrics["cycle_macro_mae_kg"] * 1000,
        "RMSE (g)": metrics["rmse_kg"] * 1000,
        "Bias (g)": metrics["bias_kg"] * 1000,
        "Within 200 g": metrics["within_200g_rate"],
        "Target-side accuracy": metrics["target_side_accuracy"],
    }
    for name, metrics in manifest["candidate_metrics"].items()
]).sort_values("Cycle-macro MAE (g)")
comparison.round({"MAE (g)": 0, "Cycle-macro MAE (g)": 0, "RMSE (g)": 0, "Bias (g)": 0, "Within 200 g": 3, "Target-side accuracy": 3})

,Candidate,MAE (g),Cycle-macro MAE (g),RMSE (g),Bias (g),Within 200 g,Target-side accuracy
4,ridge_regression,172.0,170.0,232.0,7.0,0.653,0.863
5,random_forest,176.0,178.0,238.0,-12.0,0.661,0.839
3,historical_remaining_gain,178.0,182.0,242.0,3.0,0.653,0.847
6,gradient_boosting,206.0,209.0,268.0,-20.0,0.589,0.831
0,historical_day35_mean,210.0,213.0,276.0,4.0,0.516,0.839
1,target_curve_ratio,322.0,327.0,382.0,-272.0,0.347,0.863
2,recent_linear_adg,432.0,445.0,541.0,-356.0,0.306,0.847


In [6]:
cycle_performance = pd.DataFrame.from_dict(manifest["selected_metrics"]["cycle"], orient="index")
cycle_performance.index.name = "Held-out cycle"
cycle_performance.assign(
    mae_g=cycle_performance.mae_kg * 1000,
    rmse_g=cycle_performance.rmse_kg * 1000,
    bias_g=cycle_performance.bias_kg * 1000,
)[["rows", "mae_g", "rmse_g", "bias_g", "within_200g_rate", "target_side_accuracy"]].round(2)

,rows,mae_g,rmse_g,bias_g,within_200g_rate,target_side_accuracy
Held-out cycle,,,,,,
2025-2,12,173.58,189.50,-152.69,0.58,0.08
2025-3,20,180.26,249.48,-28.26,0.75,0.80
2025-4,20,76.50,97.17,5.00,0.95,1.00
2025-5,24,294.06,358.56,69.59,0.29,1.00
2026-1,24,161.83,204.47,-50.39,0.62,0.92
2026-2,24,131.67,171.46,111.09,0.75,1.00


In [7]:
selected = manifest["selected_metrics"]
print(f"Selected held-out MAE: {selected['mae_kg']*1000:.0f} g")
print(f"Selected held-out RMSE: {selected['rmse_kg']*1000:.0f} g")
print(f"Held-out bias: {selected['bias_kg']*1000:+.0f} g")
print(f"Within 200 g: {selected['within_200g_rate']:.1%}")
print(f"Correct side of 1,800 g: {selected['target_side_accuracy']:.1%}")
print(f"Historical target hits: {manifest['actual_target_hits']} of {manifest['training_building_cycles']}")

Selected held-out MAE: 172 g
Selected held-out RMSE: 232 g
Held-out bias: +7 g
Within 200 g: 65.3%
Correct side of 1,800 g: 86.3%
Historical target hits: 5 of 31


**Interpretation:** Ridge is selected because it has the best cycle-macro MAE and remains simple and explainable. The target-side percentage looks high partly because 26 of 31 historical outcomes are below 1,800 g; the model recognizes below-target outcomes much better than the five hits.

## 4. What the selected model relies on

In [8]:
importance = pd.DataFrame(manifest["ridge_feature_importance"])
importance.head(10).rename(columns={
    "feature": "Input",
    "coefficient_kg_per_standard_deviation": "Weight change for +1 SD (kg)",
    "absolute_importance_pct": "Share of absolute reliance (%)",
    "direction": "Direction",
}).round(4)

,Input,Weight change for +1 SD (kg),Share of absolute reliance (%),Direction
0,current_to_target_ratio,0.0733,31.4221,Raises projection
1,weight_day_14_kg,0.0533,22.8404,Raises projection
2,recent_adg_kg_day,-0.0265,11.3653,Lowers projection
3,weight_day_21_kg,0.0256,10.9925,Raises projection
4,cumulative_adg_kg_day,0.0222,9.5220,Raises projection
5,weight_day_7_kg,0.0137,5.8966,Raises projection
6,current_weight_kg,-0.0079,3.3847,Lowers projection
7,has_recent_adg,-0.0061,2.6342,Lowers projection
8,measurement_day,-0.0043,1.8329,Lowers projection
9,weight_day_28_kg,0.0003,0.1094,Raises projection


These are standardized Ridge coefficients, not causal effects. Weight features are correlated, so a counterintuitive sign for one variable does not mean management should reverse it; use the overall forecast and recorded operational evidence.

## 5. Day 14 held-out proof and one complete example

In [9]:
def cycle_bootstrap_mae(frame, error_column, repeats=5000, seed=42):
    # Bootstrap whole cycles, never individual rows, to preserve grouped evidence.
    rng = np.random.default_rng(seed)
    grouped = {cycle: group for cycle, group in frame.groupby("cycle_id")}
    cycles = np.array(list(grouped))
    estimates = []
    for _ in range(repeats):
        selected = rng.choice(cycles, size=len(cycles), replace=True)
        errors = np.concatenate([grouped[cycle][error_column].to_numpy(float) for cycle in selected])
        estimates.append(np.mean(np.abs(errors)))
    return np.quantile(estimates, [0.025, 0.975])

In [10]:
day14 = pd.DataFrame(manifest["day14_backtest"])
day14["error_g"] = day14["error_kg"] * 1000
ci = cycle_bootstrap_mae(day14, "error_g")
metrics = manifest["day14_backtest_metrics"]
print(f"Day 14 building outcomes: {metrics['building_cycles']}")
print(f"Day 14 MAE: {metrics['mae_kg']*1000:.0f} g")
print(f"Cycle-bootstrap 95% interval for Day 14 MAE: {ci[0]:.0f} to {ci[1]:.0f} g")
example = day14.iloc[0]
print("\nExample")
print(f"Cycle/building: {example.cycle_id} / {example.building_id}")
print(f"Day 14 measured weight: {example.current_weight_kg*1000:.0f} g")
print(f"Projected Day 35 weight: {example.predicted_day35_weight_kg*1000:.0f} g")
print(f"Recorded Day 35 weight: {example.actual_day35_weight_kg*1000:.0f} g")
print(f"Error = projected - recorded: {example.error_g:+.0f} g")
day14.head(8)[["cycle_id", "building_id", "current_weight_kg", "predicted_day35_weight_kg", "actual_day35_weight_kg", "error_g"]]

Day 14 building outcomes: 31
Day 14 MAE: 167 g
Cycle-bootstrap 95% interval for Day 14 MAE: 125 to 220 g

Example
Cycle/building: 2025-2 / Tags 1
Day 14 measured weight: 338 g
Projected Day 35 weight: 1693 g
Recorded Day 35 weight: 1810 g
Error = projected - recorded: -117 g


,cycle_id,building_id,current_weight_kg,predicted_day35_weight_kg,actual_day35_weight_kg,error_g
0,2025-2,Tags 1,0.33800,1.692724,1.81000,-117.275720
1,2025-2,Tags 2,0.34100,1.700306,1.81000,-109.693674
2,2025-2,Tags 3,0.43400,1.935350,1.81000,125.349781
3,2025-3,Lags 1,0.25550,1.489839,1.48000,9.839103
4,2025-3,Lags 2,0.21000,1.373918,1.81000,-436.082324
5,2025-3,Tags 1,0.27758,1.606266,1.47724,129.026492
6,2025-3,Tags 2,0.29556,1.646717,1.49024,156.476503
7,2025-3,Tags 3,0.31623,1.703392,1.58210,121.291647


## 6. Historical remaining gain—why it remains in the comparison

For every eligible training building at a checkpoint age:

`remaining gain = observed Day 35 weight − checkpoint weight`

The baseline averages those gains in the training cycles and adds the average to the current weight. During validation, the held-out cycle is excluded. It is a transparent benchmark and fallback—not the live champion while Ridge remains better.

In [11]:
remaining = pd.DataFrame({
    "Checkpoint day": [int(day) for day in manifest["remaining_gain_by_measurement_day_kg"] if int(day) < 35],
    "Average historical remaining gain (g)": [gain * 1000 for day, gain in manifest["remaining_gain_by_measurement_day_kg"].items() if int(day) < 35],
}).sort_values("Checkpoint day")
remaining.round(0)

,Checkpoint day,Average historical remaining gain (g)
0,7,1406.0
1,14,1255.0
2,21,971.0
3,28,598.0


## 7. Why SMOTE or oversampling is not used

- This is regression, and standard SMOTE is a classification method.
- The 124 checkpoint rows come from only 31 independent building outcomes. Duplicating or synthesizing rows would not create new flocks.
- Synthetic weight paths may violate biological growth and can make validation look falsely precise.
- The class-like target imbalance is reported explicitly instead of hidden.

**Safer strategy:** regularized Ridge, simple baselines, complete-cycle holdouts, checkpoint/horizon metrics, cycle-level bootstrap intervals, and more standardized Day 35 outcomes over time. A hierarchical model can be considered later, after more cycles—not as a capstone requirement.

## 8. Defense takeaway

Canary's weight output is a **cycle-held-out Ridge regression for observed Day 35 average weight**, trained on 31 historical building outcomes and their earlier checkpoints. Overall held-out MAE is about 172 g; at Day 14 it is about 167 g. This is useful directional decision support, not a guarantee that a building will hit 1,800 g.